# Noise-source temperature from the 2026-10-08 interim bench run

**Goal:** measure $T_\mathrm{NS}$, the noise diode's excess temperature, with a hot load of known temperature switched through the matrix. Memo 004 currently uses the nameplate 917 K.

**Method (protocol § 11):** for every switch cycle, form the ratio $Q = (P_\mathrm{hot} - P_\mathrm{ref}) / (P_\mathrm{NON} - P_\mathrm{ref})$: how far the hot load's power sits above the reference's, in units of the noise diode's step above the reference. The receiver's gain and its own noise cancel in this ratio. If both loads reach the receiver the same way, $Q = (T_\mathrm{hot} - T_\mathrm{ref}) / T_\mathrm{NS}$: a straight line with slope $1/T_\mathrm{NS}$. **No mismatch correction is applied yet**, so every $T_\mathrm{NS}$ here is preliminary.

**Which cycles:** the hot-load thermistor lags the load while the heater ramps (~4–5 min after each new target, then overshoot). A cycle counts as *steady* when the thermistor changes by less than `bt.STEADY_K_PER_MIN` (1 K/min, what the current hardware allows); only steady cycles are fitted. Heating and cooling cycles are fitted separately as a check: if lag doesn't matter, their slopes agree with the steady one.

**Setup** (`../bench_runs/2026-10-08_interim/README.md`): hot load on the Amb/Hot port (`RFAMB`, thermistor `tempctrl_load`); ambient reference on the Feed port (`RFANT`, thermistor `tempctrl_lna2`); receiver = antenna `box-air` (input 0). Noise-diode board thermistor: `rfswitch_therm.temp_therm2`.

**Caveats:** times are UTC (the run started 2026-10-08 ~18:03 PDT, 2026-10-09 01:03 UTC). Thermistor pull-ups were measured 2-wire (~0.1–0.3 K). Single-frequency plots avoid the FM band (88–108 MHz).

Data loading, dwells and spectra come from `eigsep_data`; the code specific to this measurement is in `bench_tns.py` (edit it there and re-run; autoreload picks up changes).

## Model and sources

**Power model.** Memo 004 § 3, the minimal tier of `eigsep_cal` memo M003 § 7.1: each switch state's power is
$$P_s = g\,(M_s T_s + T_r),$$
with $g$ the receiver gain, $T_r$ the receiver's noise (assumed the same in every state, i.e. no receiver noise waves), $T_s$ the source temperature at P and $M_s$ its mismatch factor.

**Mismatch factor.** `eigsep_cal.dicke.mismatch`: $M_s = (1-|\Gamma_s|^2)/|1-\Gamma_s\Gamma_\mathrm{rec}|^2$, with both Γs at P from `eigsep_cal.S11`.

**Switched ratio Q.** The receiver calibration protocol (§ 1, § 10–11) and `eigsep_cal/docs/api.md` § 4.5 ("the switched ratio $Q_s$"); it is the fraction in the three-state Y factor (memo 004 eq. 1, Monsalve et al. 2017 eq. 1), so it is computed with `eigsep_cal.dicke.calc_Tant_star` with $T_\mathrm{NS}=1$ and a 0 K reference:
$$Q = \frac{P_\mathrm{hot}-P_\mathrm{ref}}{P_\mathrm{NON}-P_\mathrm{ref}}.$$

**T_NS (derived here from the model above).** Substituting the model, $g$ and $T_r$ cancel:
$$Q = \frac{M_\mathrm{hot}T_\mathrm{hot} - M_\mathrm{ref}T_\mathrm{ref}}{T_\mathrm{NS}},\qquad T_\mathrm{NS} \equiv \frac{P_\mathrm{NON}-P_\mathrm{ref}}{g}.$$
So $T_\mathrm{NS}$ is 1/slope of Q against $X = M_\mathrm{hot}T_\mathrm{hot} - M_\mathrm{ref}T_\mathrm{ref}$ (with $M=1$, the uncorrected $X = T_\mathrm{hot}-T_\mathrm{ref}$). This derivation has not been checked against M003's full treatment.

**What T_NS means, and the noise source's mismatch.** $T_\mathrm{NS}$ is the noise source's excess over the reference *as this receiver takes it* at P (memo 004's definition, with the bench reference in place of the field's ambient load). The noise source's own mismatch is **not corrected**: its switch paths (`VNANON`, `RFNON`) cannot be measured, so its Γ at P is unknown and its mismatch factor is absorbed into $T_\mathrm{NS}$. That factor depends on $\Gamma_\mathrm{rec}$, and the bench receiver is lab LNA 14, not the field receiver, so **this $T_\mathrm{NS}$ does not transfer directly to the field calibration**.

**Not included yet:** loss in the switch paths (their available gain), receiver noise waves.

In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import bench_tns as bt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})
CS = {bt.REF: "#0072B2", bt.HOT: "#D55E00", bt.NON: "#009E73", bt.NOFF: "#CC79A7"}   # colour-blind safe
CK = {"steady": "#000000", "heating": "#D55E00", "cooling": "#56B4E9"}
NAMEPLATE = 290 * 10 ** 0.5     # 35 dB ENR behind a 30 dB pad
to_dt = lambda t: pd.to_datetime(t, unit="s")

## 1. Load the data

In [ ]:
idx = bt.index()
allsel = idx.select()
allb = allsel.load_bundle(antenna=bt.ANTENNA, products=[])
allmeta = bt.add_temperatures(allsel.meta.reset_index(drop=True))
freqs = np.asarray(allb.freqs_mhz)
print(allb.summary())

state = {s: bt.load_state(idx, s) for s in bt.STATES}            # meta, freqs, spec, bundle
dwells = {s: bt.dwell_means(m, sp) for s, (m, f, sp, b) in state.items()}
cyc, cp = bt.cycles(dwells)

print(f"{len(allmeta)} integrations of {allmeta.integration_time.iloc[0]:.3f} s, "
      f"{to_dt(allmeta.time_best.min()):%Y-%m-%d %H:%M} to {to_dt(allmeta.time_best.max()):%H:%M} UTC")
print("switch states:", allmeta.rfswitch.value_counts().to_dict())
spec_all = np.asarray(allb.data, float)
print(f"wrapped (negative) samples: {(spec_all < 0).sum()} of {spec_all.size}")
print("dwells:", {s: len(d[0]) for s, d in dwells.items()})
print(f"complete cycles: {len(cyc)}")

## 2. Temperatures, and which cycles are steady

The threshold is `bt.STEADY_K_PER_MIN`; change it (here or in `bench_tns.py`) and re-run from this cell down.

In [ ]:
cyc = bt.classify(cyc)
print(f"threshold {bt.STEADY_K_PER_MIN} K/min:", cyc.kind.value_counts().to_dict())

fig, (a0, a1) = plt.subplots(2, 1, figsize=(8, 5), sharex=True, height_ratios=(3, 2))
t = to_dt(allmeta.time_best)
a0.plot(t, allmeta.T_hot, color=CS[bt.HOT], lw=0.8, label="hot load (tempctrl_load)")
a0.plot(t, allmeta.T_ref, color=CS[bt.REF], lw=0.8, label="ambient reference (tempctrl_lna2)")
a0.plot(t, allmeta.T_diode_board, color=CS[bt.NON], lw=0.8, label="noise-diode board (temp_therm2)")
on = allmeta.tempctrl_load_enabled.fillna(False).astype(bool).values
a0.fill_between(t, 0, 1, where=on, transform=a0.get_xaxis_transform(), color="0.88", label="heater enabled")
a0.set(ylabel="temperature (K)")
a0.legend(fontsize=7, ncol=2)
for k, c in CK.items():
    m = (cyc.kind == k).values
    a1.plot(to_dt(cyc.t[m]), cyc.rate_K_per_min[m], "o", ms=3, color=c, label=f"{k} ({m.sum()})")
a1.axhspan(-bt.STEADY_K_PER_MIN, bt.STEADY_K_PER_MIN, color="0.85", label="steady band")
a1.set_yscale("symlog", linthresh=bt.STEADY_K_PER_MIN)
a1.set(ylabel="hot-load dT/dt (K/min)", xlabel="UTC")
a1.legend(fontsize=7, ncol=4)
a1.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
plt.tight_layout()
plt.show()

**Figure 1.** Top: physical temperatures of the hot load, the ambient reference and the noise-diode board per integration, with heater-enabled periods shaded. Bottom: the hot-load thermistor's rate of change per switch cycle (symlog), coloured by how the cycle is classified; cycles inside the grey band are *steady* and are the only ones fitted. The heating ramps (~4–5 min after each new target) and the start of the cool-down are excluded.

## 3. Is the switch sequence clean?

In [ ]:
k = allmeta.index[allmeta.time_best > allmeta.time_best.iloc[0] + 600][0]     # three cycles, 10 min in
s = slice(k, k + 3 * 60)
i150 = np.argmin(abs(freqs - 150))
fig, (a0, a1) = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
for st, c in CS.items():
    m = (allmeta.rfswitch.iloc[s] == st).values
    a0.semilogy(t[s][m], spec_all[s][m, i150], "o", ms=3, color=c, label=st)
a0.set(ylabel=f"power at {freqs[i150]:.1f} MHz (counts)")
a0.legend(fontsize=7, ncol=4)
a1.plot(t[s], allmeta.T_diode_board.iloc[s], color=CS[bt.NON])
m = (allmeta.rfswitch.iloc[s] == bt.NON).values
a1.plot(t[s][m], allmeta.T_diode_board.iloc[s][m], "o", ms=3, color=CS[bt.NON], label="RFNON rows")
a1.set(ylabel="diode board (K)", xlabel="UTC")
a1.legend(fontsize=7)
a1.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
plt.tight_layout()
plt.show()

**Figure 2.** Three switch cycles: raw power per integration at 150 MHz coloured by switch state (top), and the noise-diode board thermistor (bottom). Integrations whose state is UNKNOWN (switching mid-integration) are not plotted and never used. The board thermistor rises during each `RFNON` dwell: the diode heats when on, which confirms `temp_therm2` is the diode's thermistor on this board.

## 4. Settling after a switch change

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
b = (freqs > 60) & (freqs < 240) & ~((freqs > 88) & (freqs < 108))
for st, c in CS.items():
    meta, f, sp, bb = state[st]
    rel = []
    for d, g in meta[meta.dwell >= 0].groupby("dwell"):
        x = np.nanmean(sp[g.index][:, b], axis=1)
        rel.append(pd.Series(x / x[-3:].mean() - 1, index=g.pos_in_dwell.values))
    r = pd.concat(rel, axis=1)
    ax.plot(r.index, r.mean(axis=1) * 100, "o-", color=c, ms=3, label=st)
ax.set(xlabel="integration within dwell", ylabel="power vs dwell end (%)")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

**Figure 3.** Band power (60–240 MHz without FM) of each integration relative to the last three integrations of its dwell, averaged over dwells, per state. A first point that sits off the rest means the switch had not settled; then pass `skip_first=1` to `bt.dwell_means`.

## 5. Within-dwell noise against the radiometer equation

In [ ]:
meta, f, sp, bb = state[bt.REF]
i = np.argmin(abs(freqs - 150))
sd, pred = [], []
for d, g in meta[meta.dwell >= 0].groupby("dwell"):
    x = sp[g.index, i]
    x = x[x > 0]
    if len(x) > 3:
        sd.append(x.std(ddof=1)); pred.append(x.mean() / np.sqrt(bt.ENBW_HZ * g.integration_time.iloc[0]))
r = np.array(sd) / np.array(pred)
print(f"RFANT dwells at {freqs[i]:.1f} MHz: within-dwell scatter / radiometer prediction = {r.mean():.2f} "
      f"(95th pct {np.percentile(r, 95):.2f}, max {r.max():.2f}), Δν = {bt.ENBW_HZ/1e3:.1f} kHz, τ = {meta.integration_time.iloc[0]:.3f} s")

The scatter of raw power between the integrations of one ~10 s dwell, divided by $P/\sqrt{\Delta\nu\,\tau}$. Near 1 means the data are radiometer-noise-limited on that timescale. This is not the same as the cycle-to-cycle scatter of Q in § 9, which covers minutes and includes thermistor noise and drift.

## 6. Q against the temperature difference

In [ ]:
Q = bt.q_hot(cp)
dT = (cyc.T_hot - cyc.T_ref).values
fits = {k: bt.fit_q(dT, Q, (cyc.kind == k).values) for k in CK}
fs = fits["steady"]
picks = [55, 75, 150, 220]
fig, axs = plt.subplots(2, len(picks), figsize=(10, 4.4), sharex=True, height_ratios=(2, 1))
xx = np.array([dT.min(), dT.max()])
for j, fr in enumerate(picks):
    i = np.argmin(abs(freqs - fr))
    for kind, c in CK.items():
        m = (cyc.kind == kind).values
        axs[0, j].plot(dT[m], Q[m, i], ".", ms=3, color=c, label=kind)
        axs[1, j].plot(dT[m], Q[m, i] - fs["a"][i] - fs["b"][i] * dT[m], ".", ms=2, color=c)
    axs[0, j].plot(xx, fs["a"][i] + fs["b"][i] * xx, "k-", lw=0.8)
    axs[0, j].set_title(f"{freqs[i]:.1f} MHz: T_NS = {1/fs['b'][i]:.0f} K (steady)", fontsize=8)
    axs[1, j].axhline(0, color="k", lw=0.5)
    axs[1, j].set_xlabel("T_hot − T_ref (K)")
axs[0, 0].set_ylabel("Q"); axs[1, 0].set_ylabel("Q − steady fit")
axs[0, 0].legend(fontsize=7)
plt.tight_layout()
plt.show()

**Figure 4.** Q per switch cycle against the thermistor temperature difference at four frequencies, coloured by cycle kind, with the straight-line fit to the steady cycles (top) and every cycle's residual from it (bottom). The slope is $1/T_\mathrm{NS}$ (uncorrected for mismatch). Heating and cooling cycles off the steady line are what thermistor lag does; a non-zero intercept means the two load paths differ in loss or mismatch.

### The ramp check
Before looking: if the thermistor tracked the load during ramps, heating and cooling cycles would give the same slope as the steady ones, within errors. A slope differing by several σ says lag matters and justifies excluding them.

In [ ]:
rows = []
for fr in picks:
    i = np.argmin(abs(freqs - fr))
    row = {"MHz": round(freqs[i], 1)}
    for k in CK:
        g = fits[k]
        row[f"T_NS {k} (K)"] = round(1 / g["b"][i])
        if k != "steady":
            row[f"{k} − steady (σ)"] = round((g["b"][i] - fs["b"][i]) / np.hypot(g["sb"][i], fs["sb"][i]), 1)
    rows.append(row)
print({k: g["n"] for k, g in fits.items()}, "cycles fitted per kind")
pd.DataFrame(rows)

## 7. $T_\mathrm{NS}$ across the band (steady cycles, uncorrected)

In [ ]:
tns = 1 / fs["b"]
band = (freqs > 45) & (freqs < 250)
fm = (freqs > 88) & (freqs < 108)
fig, (a0, a1) = plt.subplots(2, 1, figsize=(8, 4.8), sharex=True, height_ratios=(2, 1))
a0.plot(freqs[band], tns[band], color="k", lw=0.7, label="bench, steady cycles, no mismatch correction")
a0.axhline(NAMEPLATE, color=CS[bt.HOT], ls="--", lw=0.8, label=f"nameplate {NAMEPLATE:.0f} K")
a0.axvspan(88, 108, color="0.9", label="FM band")
a0.set(ylabel="T_NS (K)", ylim=(0, 1200))
a0.legend(fontsize=7)
a1.plot(freqs[band], fs["a"][band], color="0.3", lw=0.7)
a1.axhline(0, color="k", lw=0.5)
a1.axvspan(88, 108, color="0.9")
a1.set(ylabel="intercept of Q", xlabel="frequency (MHz)")
plt.tight_layout()
plt.show()
for lo, hi in [(50, 88), (108, 150), (150, 200), (200, 250)]:
    m = (freqs >= lo) & (freqs < hi) & np.isfinite(tns)
    print(f"{lo}-{hi} MHz: T_NS mean {tns[m].mean():.0f} K, range {tns[m].min():.0f}-{tns[m].max():.0f} K, "
          f"slope error mean {100*np.mean(fs['sb'][m]/fs['b'][m]):.1f}%")

**Figure 5.** $T_\mathrm{NS}$ per channel from the slope of Q against $T_\mathrm{hot} - T_\mathrm{ref}$ over the steady cycles (top, nameplate dashed, FM band shaded), and the fit's intercept (bottom). No mismatch correction: the hot and reference loads sit on different switch paths with different reflections at P, which this ignores. No channels are masked here; sections 8–10 add the mismatch correction and the mask.

## 8. Reflections at P, and a check on them

The sweeps are calibrated with `eigsep_cal.S11` and its packaged 2026 characterizations (internal standards, switch paths). This is the field switch board, the one the packaged switch paths were characterized on; still, first a check that could fail: a 50 Ω load de-embedded to its own port (@DUT) must have $|\Gamma| \approx 0$. If the packaged characterization did not fit the board as it is now, it would not.

In [ ]:
s11 = bt.load_s11()
fv = np.asarray(s11.freqs_mhz)
fig, axs = plt.subplots(1, 3, figsize=(10, 3.2), sharey=True)
for ax, (st, dut) in zip(axs[:2], bt.S11_DUT.items()):
    for plane, ls in (("dut", ":"), ("lna", "-")):
        ts, g = s11.get_all_s11s(dut, plane)
        for x in g:
            ax.semilogy(fv, np.abs(x), ls, color=CS[st], lw=0.6, alpha=0.7)
        ax.plot([], [], ls, color=CS[st], label=f"@{'DUT' if plane == 'dut' else 'P'} ({len(g)} sweeps)")
    ax.set(title=f"{st} ({'hot load' if st == bt.HOT else 'ambient reference'})", xlabel="frequency (MHz)")
    ax.legend(fontsize=7)
ts, g = s11.get_all_s11s("rec")
for x in g:
    axs[2].semilogy(fv, np.abs(x), color="0.3", lw=0.6, alpha=0.7)
axs[2].set(title=f"receiver at P ({len(g)} sweeps)", xlabel="frequency (MHz)")
axs[0].set_ylabel("|Γ|")
for ax in axs:
    ax.axvspan(bt.BAND_MHZ[0], bt.BAND_MHZ[1], color="0.93", zorder=0)
plt.tight_layout()
plt.show()
for st, dut in bt.S11_DUT.items():
    ts, g = s11.get_all_s11s(dut, "dut")
    b = (fv >= bt.BAND_MHZ[0]) & (fv <= bt.BAND_MHZ[1])
    print(f"{st} at @DUT, {bt.BAND_MHZ[0]:.0f}-{bt.BAND_MHZ[1]:.0f} MHz: max |Γ| over all sweeps {np.abs(g[:, b]).max():.4f}")

**Figure 6.** $|\Gamma|$ of the hot load (`RFAMB`) and the reference (`RFANT`) at their own ports (@DUT, dotted) and at plane P through their switch paths (solid), and of the receiver at P, for every sweep of the run; the band used is shaded. The loads at @DUT sit near zero, which is the check passing: the packaged switch-path and internal-standard characterizations fit this board. At P the switch paths raise both loads to $|\Gamma|$ of order 0.05–0.1, and the receiver is far from matched.

In [ ]:
t_cyc = cyc.t.to_numpy()
M, gam, grec = bt.mismatch_factors(s11, freqs, t_cyc)
inb = (freqs >= bt.BAND_MHZ[0]) & (freqs <= bt.BAND_MHZ[1])
steady = (cyc.kind == "steady").values
fig, ax = plt.subplots(figsize=(8, 3))
for st in (bt.HOT, bt.REF):
    ax.plot(freqs[inb], M[st][steady][:, inb].mean(0), color=CS[st], lw=0.8, label=f"M {st}")
ax.plot(freqs[inb], (M[bt.HOT] - M[bt.REF])[steady][:, inb].mean(0) + 1, color="0.4", lw=0.8, ls="--", label="1 + (M_hot − M_ref)")
ax.set(xlabel="frequency (MHz)", ylabel="mismatch factor")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

**Figure 7.** Mismatch factors $M = (1-|\Gamma|^2)/|1-\Gamma\,\Gamma_\mathrm{rec}|^2$ of the hot load and the reference at P, averaged over the steady cycles (Γs interpolated in time between sweeps and onto the correlator channels). Each is within ~5 % of 1, but they differ from each other by up to a few per cent, which is what biased the uncorrected slope.

## 9. Channel mask

The corrected abscissa is $X = M_\mathrm{hot} T_\mathrm{hot} - M_\mathrm{ref} T_\mathrm{ref}$, and the fit is Q against X over the steady cycles, per channel. A channel is masked for any of four reasons (`bt.channel_mask`): outside the band; fit residuals, in units of radiometer noise, with a mean square above 3× the in-band median (drift or interference); a slope error above 5 % (weak noise-source step); or $T_\mathrm{NS}$ more than 5 robust σ from its running median. Medians appear only here, to find bad channels.

In [ ]:
TH = cyc.T_hot.to_numpy()[:, None]; TR = cyc.T_ref.to_numpy()[:, None]
X_unc = np.repeat(TH - TR, freqs.size, axis=1)
X_cor = M[bt.HOT] * TH - M[bt.REF] * TR
fit_unc = bt.fit_q(X_unc, Q, steady)
fit_cor = bt.fit_q(X_cor, Q, steady)
tau = allmeta.integration_time.iloc[0]
sq = bt.sigma_q(cyc, cp, tau)
mask, chi2 = bt.channel_mask(freqs, fit_cor, sq, steady)
print({k: int(v.sum()) for k, v in mask.items()}, "channels masked, of", freqs.size)
print(f"in-band median chi2 {np.nanmedian(chi2[inb]):.2f}; FM channels masked: {int((mask['any'] & (freqs > 88) & (freqs < 108)).sum())} of {int(((freqs > 88) & (freqs < 108)).sum())}")

fig, axs = plt.subplots(3, 1, figsize=(8, 6.5), sharex=True)
med = np.nanmedian(chi2[inb])
axs[0].semilogy(freqs[inb], chi2[inb], ".", ms=2, color="0.4")
axs[0].semilogy(freqs[mask["chi2"]], chi2[mask["chi2"]], ".", ms=3, color="#D55E00", label=f"masked ({mask['chi2'].sum()})")
axs[0].axhline(3 * med, color="k", lw=0.8, ls="--", label="3 × in-band median")
axs[0].set(ylabel="cycle-to-cycle mean (residual/σ_Q)²")
rel = fit_cor["sb"] / fit_cor["b"]
axs[1].semilogy(freqs[inb], np.abs(rel[inb]), ".", ms=2, color="0.4")
axs[1].semilogy(freqs[mask["rel_err"]], np.abs(rel[mask["rel_err"]]), ".", ms=3, color="#D55E00", label=f"masked ({mask['rel_err'].sum()})")
axs[1].axhline(0.05, color="k", lw=0.8, ls="--", label="5 %")
axs[1].set(ylabel="slope error / slope")
tns_all = 1 / fit_cor["b"]
keep = inb & ~mask["chi2"] & ~mask["rel_err"]
axs[2].plot(freqs[keep], tns_all[keep], ".", ms=2, color="0.4")
axs[2].plot(freqs[mask["outlier"]], tns_all[mask["outlier"]], ".", ms=3, color="#D55E00", label=f"masked outliers ({mask['outlier'].sum()})")
axs[2].set(ylabel="T_NS, corrected (K)", ylim=(300, 800), xlabel="frequency (MHz)")
for ax in axs:
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

**Figure 8.** The three data-driven mask tests per channel in the band, with each threshold dashed and the masked channels in orange (counts in the legends): top, the mean square of the fit residuals in radiometer-noise units; middle, the slope's relative error; bottom, the corrected $T_\mathrm{NS}$ of the channels that pass the first two, with the outliers from its running median marked. The test statistic is cycle-to-cycle: the fit residuals of Q between switch cycles, in units of the radiometer noise predicted for Q (`bt.sigma_q`, the radiometer equation propagated through Q's definition). An in-band median well above 1 means Q varies between cycles by more than radiometer noise, although within a dwell the data are noise-limited (§ 5); thermistor noise is the likely cause (it enters as slope × temperature error).

## 10. $T_\mathrm{NS}$, mismatch-corrected and masked

In [ ]:
good = ~mask["any"]
fig, (a0, a1) = plt.subplots(2, 1, figsize=(8, 4.8), sharex=True, height_ratios=(2, 1))
for name, ft, c in (("uncorrected", fit_unc, "0.6"), ("mismatch-corrected", fit_cor, "k")):
    y = np.where(good, 1 / ft["b"], np.nan)
    a0.plot(freqs, y, color=c, lw=0.8, label=name)
    a1.plot(freqs, np.where(good, ft["a"], np.nan), color=c, lw=0.8)
a0.axhline(NAMEPLATE, color=CS[bt.HOT], ls="--", lw=0.8, label=f"nameplate {NAMEPLATE:.0f} K")
a0.set(ylabel="T_NS (K)", ylim=(0, 1000), xlim=bt.BAND_MHZ)
a0.legend(fontsize=7)
a1.axhline(0, color="k", lw=0.5)
a1.set(ylabel="intercept of Q", xlabel="frequency (MHz)")
plt.tight_layout()
plt.show()
rows = []
for lo, hi in [(50, 88), (108, 150), (150, 200), (200, 245)]:
    k = (freqs >= lo) & (freqs < hi) & good
    for name, ft in (("uncorrected", fit_unc), ("corrected", fit_cor)):
        t_ = 1 / ft["b"][k]; w = 1 / ft["sb"][k] ** 2
        rows.append({"band (MHz)": f"{lo}-{hi}", "fit": name, "channels": int(k.sum()),
                     "T_NS weighted mean (K)": round(np.sum(w * t_) / w.sum()),
                     "min (K)": round(t_.min()), "max (K)": round(t_.max()),
                     "mean intercept": round(ft["a"][k].mean(), 4)})
pd.DataFrame(rows)

**Figure 9.** $T_\mathrm{NS}$ per channel from the steady cycles, without (grey) and with (black) the mismatch correction, masked channels blank; the fit's intercept below. The table gives inverse-variance weighted means per band with the per-channel range.

**What $T_\mathrm{NS}$ means here:** the noise source's excess over the reference, as *this* receiver takes it at P, $(P_\mathrm{NON} - P_\mathrm{ref})/g$. The noise source's own mismatch is absorbed, not corrected, so the number is tied to the bench receiver (lab LNA 14) and does not transfer directly to the field (see Model and sources).

## Next steps
1. A bench receiver that is a better proxy for the field receiver (or the field receiver itself), so that the noise source's absorbed mismatch transfers.
2. Loss in the switch paths: their available gain moves each load's temperature at P toward the switch board's (`eigsep_cal`'s `embed`, api.md § 4.3).
3. The intercepts: what remains after the mismatch correction.
4. Why Q scatters more than radiometer noise between cycles (thermistor noise?).
5. $T_\mathrm{NS}$ against the diode-board temperature.
6. Uncertainty from the thermistor calibration (~0.1–0.3 K) alongside the fit error.

## 11. SNAP warm-up

The SNAP's FPGA temperature (`fpga_temp.temp_c`, logged about every 20 s) was still rising at the start of the run, and the reference load's power rose with it while the load itself cooled. A slow gain change multiplies all four states in a cycle by the same factor and cancels in Q, so it should not bias $T_\mathrm{NS}$. The check: refit with the first 30 and 60 minutes left out. If the warm-up mattered, $T_\mathrm{NS}$ would move by more than its error.

In [ ]:
snap = {st: bt.snap_temperature(idx, dwells[st][0].t.to_numpy()) for st in (bt.REF, bt.NOFF)}
bmid = (freqs > 108) & (freqs < 200) & ~mask["any"]
fig, axs = plt.subplots(3, 1, figsize=(8, 5.5), sharex=True)
td = {st: to_dt(dwells[st][0].t) for st in (bt.REF, bt.NOFF)}
axs[0].plot(td[bt.REF], snap[bt.REF], color="0.3")
axs[0].set(ylabel="SNAP FPGA (°C)")
t_end = dwells[bt.REF][0].t.max()
for st in (bt.REF, bt.NOFF):
    p = np.nanmean(dwells[st][1][:, bmid], axis=1)
    last = dwells[st][0].t.to_numpy() > t_end - 3600
    axs[1].plot(td[st], p / np.mean(p[last]), ".", ms=2, color=CS[st], label=st)
# axs[1].set(ylabel="power / last-hour mean", ylim=(0.985, 1.01))
axs[1].set(ylabel="power / last-hour mean", ylim=(0.95, 1.01))
axs[1].legend(fontsize=7)
axs[2].plot(td[bt.REF], dwells[bt.REF][0].T_ref, color=CS[bt.REF])
axs[2].set(ylabel="T_ref (K)", xlabel="UTC")
axs[2].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
plt.tight_layout()
plt.show()
p = np.nanmean(dwells[bt.REF][1][:, bmid], axis=1)
print(f"corr(P_ref, SNAP temperature) = {np.corrcoef(p, snap[bt.REF])[0, 1]:.3f}; "
      f"corr(P_ref, T_ref) = {np.corrcoef(p, dwells[bt.REF][0].T_ref)[0, 1]:.3f}")

**Figure 10.** The SNAP's FPGA temperature (top); the band power (108–200 MHz, unmasked channels) of each reference (`RFANT`) and noise-off (`RFNOFF`) dwell relative to its last-hour mean (middle; the y range clips a few `RFNOFF` dwells near 01:47, 02:56 and 04:05 UTC whose power about doubles, which do not enter Q); the reference load's thermistor (bottom). The reference power rises ~1 % over the first hour with the SNAP temperature while the load cools, so the rise is in the receiver chain. That it is the SNAP and not, e.g., the LNAs warming is not established: both warm together.

In [ ]:
t0 = cyc.t.min()
rows = []
for skip in (0, 30, 60):
    use = steady & (cyc.t.values >= t0 + 60 * skip)
    ft = bt.fit_q(X_cor, Q, use)
    for lo, hi in [(50, 88), (108, 150), (150, 200), (200, 245)]:
        k = (freqs >= lo) & (freqs < hi) & ~mask["any"]
        t_ = 1 / ft["b"][k]; w = 1 / ft["sb"][k] ** 2
        rows.append({"first minutes left out": skip, "band (MHz)": f"{lo}-{hi}", "steady cycles": int(use.sum()),
                     "T_NS weighted mean (K)": round(np.sum(w * t_) / w.sum(), 1)})
tab = pd.DataFrame(rows).pivot(index="band (MHz)", columns="first minutes left out", values="T_NS weighted mean (K)")
print("steady cycles fitted:", {s: int((steady & (cyc.t.values >= t0 + 60 * s)).sum()) for s in (0, 30, 60)})
tab

Corrected $T_\mathrm{NS}$ per band with the first 0, 30 and 60 minutes of steady cycles left out. Leaving out the start also drops most of the ambient period and the 47 °C plateau, so the lever arm changes too; a shift comparable to the per-channel spread of § 10 is not evidence of a warm-up bias.